# LLM Unlearning

## What This Is
LLM unlearning addresses a different problem than classical ML unlearning: *fact erasure*. Instead of removing a data point's influence on predictions, we want to make the model forget specific memorized facts — names, addresses, copyrighted text, dangerous knowledge.

## ROME and MEMIT
**ROME (Rank-One Model Editing)** treats a fact as a key-value association stored in FFN layers. Given a factual triple (subject, relation, object), ROME modifies a specific MLP layer to change the stored association. It uses a rank-one update to the weight matrix.

**MEMIT (Mass-Editing Memory In Transformers)** extends ROME to batch edits — editing thousands of facts simultaneously by distributing the update across multiple layers.

## Unlearning vs Fine-Tuning
Fine-tuning on the forget set with random labels degrades the model broadly — it unlearns the target but also damages related knowledge. ROME/MEMIT surgical edits are more targeted: they change specific fact associations while preserving surrounding knowledge. The tradeoff is that ROME requires knowing exactly which layer stores a fact (using causal tracing).

## Limitations
LLM unlearning is a nascent field with no settled verification methodology. Key open problems:
- A model that "forgets" a fact via ROME may still have it encoded in other layers
- Regeneration attacks: prompts that elicit forgotten facts through paraphrase
- No equivalent of DP's ε guarantee for LLM unlearning

## Causal Tracing
Before editing, ROME uses *causal tracing* (activation patching) to identify which MLP layer in which transformer block stores a specific fact. It runs the model twice — once clean, once corrupted — and measures which layer restores the correct output when patched. This localizes the fact to a single MLP.

In [ ]:
import numpy as np
import json
from typing import Dict, List, Tuple, Optional

# Simulate a tiny transformer with explicit key-value storage in FFN layers
# This is a pedagogical simplification of the ROME insight

np.random.seed(42)

class TinyFactMemory:
    """Simulates an LLM's factual storage as a key-value associative memory."""
    
    def __init__(self, d_model: int = 64, n_facts: int = 20):
        self.d_model = d_model
        # Key vectors (subject representations)
        self.K = np.random.randn(n_facts, d_model) * 0.1
        self.K = self.K / np.linalg.norm(self.K, axis=1, keepdims=True)
        # Value vectors (stored facts)
        self.V = np.random.randn(n_facts, d_model) * 0.1
        # Weight matrix: W = V^T K^T / n (outer product storage)
        self.W = (self.V.T @ self.K) / n_facts
        
        # Fact database: maps string keys to indices
        self.facts: Dict[str, int] = {}
        self.n_stored = 0
    
    def store_fact(self, subject: str, subject_vec: np.ndarray, answer_vec: np.ndarray):
        """Store a fact in the weight matrix."""
        idx = self.n_stored % len(self.K)
        self.K[idx] = subject_vec / (np.linalg.norm(subject_vec) + 1e-8)
        self.V[idx] = answer_vec
        self.facts[subject] = idx
        # Update weight matrix
        self.W = (self.V.T @ self.K) / len(self.K)
        self.n_stored += 1
    
    def recall(self, subject_vec: np.ndarray) -> np.ndarray:
        """Recall the stored value for a subject query."""
        return self.W @ subject_vec
    
    def recall_similarity(self, subject_vec: np.ndarray, target_vec: np.ndarray) -> float:
        """Cosine similarity between recalled and target."""
        recalled = self.recall(subject_vec)
        n1 = np.linalg.norm(recalled)
        n2 = np.linalg.norm(target_vec)
        if n1 < 1e-8 or n2 < 1e-8:
            return 0.0
        return float(np.dot(recalled, target_vec) / (n1 * n2))

# Create fact memory with some stored facts
memory = TinyFactMemory(d_model=32, n_facts=10)

# Generate stable representations for subjects
facts = {
    'Paris capital of': ('France', np.random.randn(32)),
    'Eiffel Tower location': ('Paris', np.random.randn(32)),
    'Einstein born in': ('Ulm', np.random.randn(32)),
    'Delete target fact': ('SECRET_DATA', np.random.randn(32)),
}

answer_vecs = {k: np.random.randn(32) for k in facts}

for subject, (_, subj_vec) in facts.items():
    memory.store_fact(subject, subj_vec, answer_vecs[subject])

print('Stored facts:', list(facts.keys()))


In [ ]:
# Simulate ROME-style rank-one edit for fact erasure

def rome_edit(W: np.ndarray, k: np.ndarray, v_target: np.ndarray, 
              C: np.ndarray = None, lam: float = 0.1) -> np.ndarray:
    """
    ROME rank-one update to W to associate key k with new value v_target.
    For unlearning: v_target = random or zero vector.
    
    W_new = W + (v_target - W @ k) @ k^T / (k^T C^{-1} k)
    where C is the covariance of key activations.
    """
    if C is None:
        C = np.eye(len(k)) * lam
    
    # Current output for this key
    v_current = W @ k
    residual = v_target - v_current
    
    # Solve C^{-1} k
    C_inv_k = np.linalg.solve(C + lam * np.eye(len(k)), k)
    
    # Rank-one update
    delta_W = np.outer(residual, C_inv_k) / (k @ C_inv_k)
    return W + delta_W

# Recall accuracy BEFORE unlearning
print('Before unlearning:')
for subject, (answer, subj_vec) in facts.items():
    sim = memory.recall_similarity(subj_vec, answer_vecs[subject])
    print(f'  [{subject}] → recall similarity: {sim:.3f}')

# Unlearn 'Delete target fact'
forget_subject = 'Delete target fact'
forget_key = facts[forget_subject][1]  # subject vector
forget_key_norm = forget_key / (np.linalg.norm(forget_key) + 1e-8)

# Erasure target: random vector (not the original value)
null_value = np.random.randn(32) * 0.01

# Apply ROME edit
memory.W = rome_edit(memory.W, forget_key_norm, null_value, lam=0.5)

print('\nAfter ROME unlearning of "Delete target fact":')
for subject, (answer, subj_vec) in facts.items():
    sim = memory.recall_similarity(subj_vec, answer_vecs[subject])
    marker = ' <-- ERASED' if subject == forget_subject else ''
    print(f'  [{subject}] → recall similarity: {sim:.3f}{marker}')


In [ ]:
# Verify unlearning and collateral damage
print('Unlearning Verification:')
print('=' * 50)

before_sims = {}
after_sims = {}

# Recompute before (using original memory state conceptually)
# We already modified memory, so we compare to expected

forget_sim_after = memory.recall_similarity(
    facts[forget_subject][1], answer_vecs[forget_subject]
)

print(f'Forget fact recall similarity after erasure: {forget_sim_after:.4f}')
print(f'  (near 0 = successfully erased, near 1 = still remembered)')

print('\nCollateral damage check (other facts should be preserved):')
for subject in facts:
    if subject == forget_subject:
        continue
    sim = memory.recall_similarity(facts[subject][1], answer_vecs[subject])
    print(f'  [{subject[:30]}]: {sim:.3f}')

print('\nKey insight: ROME targets a single rank-one update')
print('minimizing collateral damage to other stored facts.')
print('MEMIT extends this to batch erasure across multiple layers.')
